# Notebook 4 — M4: Tabular Diabetes Risk (NHANES + Pima)
## Explainable Multimodal Diabetes/Metabolic Risk Framework

**This notebook:**
1. Loads and preprocesses NHANES 2017-18 tabular data
2. Trains XGBoost classifier for binary diabetes prediction
3. Runs SHAP TreeExplainer for per-feature explanations
4. Cross-validates on Pima Indians dataset
5. Extracts per-sample risk probabilities and saves to Drive
6. Generates Section 4 SHAP plots + hyperparameter table

**Runtime:** ~15 min on Colab CPU (XGBoost doesn't need GPU)

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────────────────
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib nhanes

import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/MultimodalDisease'
REPO_DIR = '/content/Multimodal_Disease'
os.environ['MMDISEASE_BASE'] = BASE_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup complete.')

In [ ]:
# ─── Load NHANES ──────────────────────────────────────────────────────────────
from src.utils.data_utils import load_nhanes, prepare_tabular_splits
from src.config import M4

NHANES_DIR = f'{BASE_DIR}/data/nhanes'
X_nhanes, y_nhanes = load_nhanes(nhanes_dir=NHANES_DIR)
print(f'NHANES loaded: {X_nhanes.shape} features, {y_nhanes.sum()} positives / {len(y_nhanes)} total')
print(f'Features: {list(X_nhanes.columns)}')

In [ ]:
# ─── Exploratory Data Analysis ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
key_features = ['age', 'bmi', 'fasting_glucose', 'hba1c',
                'systolic_bp', 'total_cholesterol', 'waist_circumference', 'hdl']

for ax, feat in zip(axes.flat, key_features):
    if feat not in X_nhanes.columns:
        ax.set_visible(False)
        continue
    for label, color, name in [(0, '#4C72B0', 'Non-diabetic'), (1, '#C44E52', 'Diabetic')]:
        subset = X_nhanes.loc[y_nhanes == label, feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=7)

plt.suptitle('NHANES Feature Distributions by Diabetes Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/m4_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# ─── Train/val/test split ─────────────────────────────────────────────────────
X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_tabular_splits(X_nhanes, y_nhanes)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test positive rate:  {y_test.mean():.3f}')

In [ ]:
# ─── Train XGBoost ────────────────────────────────────────────────────────────
from src.models.m4_tabular import train_tabular_model

M4_SAVE_PATH = f'{BASE_DIR}/saved_models/m4_tabular.pkl'

m4_model = train_tabular_model(
    X_train    = X_train,
    y_train    = y_train,
    X_val      = X_val,
    y_val      = y_val,
    model_type = M4['model_type'],
    save_path  = M4_SAVE_PATH
)
print('XGBoost training complete.')

In [ ]:
# ─── Evaluate on test set ────────────────────────────────────────────────────
from src.models.m4_tabular import extract_risk_scores as m4_extract, predict_binary
from src.utils.eval_utils import compute_metrics, print_classification_report
from src.utils.viz_utils import plot_confusion_matrix, plot_roc_curve

m4_risks = m4_extract(m4_model, X_test)
m4_preds = predict_binary(m4_model, X_test)

np.save(f'{BASE_DIR}/outputs/scores/m4_risk_scores.npy', m4_risks)
np.save(f'{BASE_DIR}/outputs/scores/m4_labels.npy', y_test.values)
print(f'M4 risk scores saved. Shape: {m4_risks.shape}')

print('\n=== M4 Metrics ===')
m4_metrics = compute_metrics(y_test, m4_preds, y_prob=m4_risks, module_name='M4 Tabular')
print_classification_report(y_test, m4_preds, class_names=['Non-Diabetic', 'Diabetic'])

plot_confusion_matrix(y_test, m4_preds, ['Non-Diabetic', 'Diabetic'], 'M4 Tabular XGBoost', save=True)
plot_roc_curve(y_test, m4_risks, 'M4 Tabular XGBoost', save=True)

In [ ]:
# ─── SHAP Explanations ────────────────────────────────────────────────────────
import shap
from src.xai.shap_explainer import TabularSHAPExplainer

SHAP_DIR = f'{BASE_DIR}/outputs/figures/m4_shap'
os.makedirs(SHAP_DIR, exist_ok=True)

shap_exp = TabularSHAPExplainer(
    model         = m4_model,
    X_background  = X_train,
    feature_names = list(X_train.columns)
)

# Compute SHAP values on test set (sample 200 for speed)
X_test_sample = X_test.sample(min(200, len(X_test)), random_state=42)
shap_values   = shap_exp.compute_shap_values(X_test_sample)

print(f'SHAP values computed. Shape: {shap_values.shape}')

In [ ]:
# ─── SHAP Summary Plot ────────────────────────────────────────────────────────
shap_exp.plot_summary(
    X_test_sample, shap_values,
    max_display=15,
    save_path=f'{SHAP_DIR}/m4_shap_summary.png'
)
plt.show()
print('SHAP summary plot saved.')

In [ ]:
# ─── SHAP Waterfall for a single high-risk patient ───────────────────────────
# Find a high-risk positive sample for illustration
high_risk_idx = m4_risks.argsort()[-1]  # highest predicted risk
X_single      = X_test.iloc[[high_risk_idx]]

shap_exp.plot_waterfall(
    X_row=X_single,
    save_path=f'{SHAP_DIR}/m4_shap_waterfall_highrisk.png'
)
plt.show()

# Also show a low-risk example
low_risk_idx  = m4_risks.argsort()[0]
shap_exp.plot_waterfall(
    X_row=X_test.iloc[[low_risk_idx]],
    save_path=f'{SHAP_DIR}/m4_shap_waterfall_lowrisk.png'
)
print('SHAP waterfall plots saved.')

In [ ]:
# ─── Feature Importance Bar Chart ────────────────────────────────────────────
from src.models.m4_tabular import get_feature_importance
from src.utils.viz_utils import plot_shap_bar

# XGBoost gain-based importances (for comparison with SHAP)
feat_imp = get_feature_importance(m4_model, list(X_train.columns))
print('Top 10 features (gain-based):')
print(feat_imp.head(10).to_string(index=False))

# SHAP-based bar chart
plot_shap_bar(
    shap_values   = shap_values,
    feature_names = list(X_test_sample.columns),
    title         = 'M4 — Mean |SHAP Value| per Feature',
    save          = True,
    filename      = 'm4_shap_bar'
)
plt.show()

In [ ]:
# ─── Cross-validation on Pima Indians ────────────────────────────────────────
from src.utils.data_utils import load_pima
from src.models.m4_tabular import validate_on_pima

PIMA_DIR = f'{BASE_DIR}/data/pima'
X_pima, y_pima = load_pima(pima_dir=PIMA_DIR)

print('\n=== Pima Cross-Validation ===')
pima_results = validate_on_pima(m4_model, X_pima, y_pima)
print('(Shows generalization beyond NHANES training distribution)')

In [ ]:
# ─── Hyperparameter tuning table (Section 4.6) ───────────────────────────────
from src.utils.eval_utils import build_hyperparam_table

hyperparam_df = build_hyperparam_table()
m4_params = hyperparam_df[hyperparam_df['Module'].str.startswith('M4')]
print('\n=== M4 Hyperparameter Tuning (Section 4.6) ===')
print(m4_params.to_string(index=False))

hyperparam_df.to_csv(f'{BASE_DIR}/outputs/hyperparameter_summary.csv', index=False)

print(f'\nNotebook 4 complete.')
print(f'  M4 model: {BASE_DIR}/saved_models/m4_tabular.pkl')
print(f'  M4 scores: {BASE_DIR}/outputs/scores/m4_risk_scores.npy')
print(f'  SHAP plots: {SHAP_DIR}/')